<a href="https://colab.research.google.com/github/saparbayev-azizbek-12/bi-and-ai-talents-dl/blob/main/lesson-24/NameRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [111]:
import unicodedata

def unicodeToAscii(s):
  return ''.join(
    c for c in unicodedata.normalize('NFD', s)
    if unicodedata.category(c) != 'Mn'
    and c in all_letters
  )

print (f"converting 'Ślusàrski' to {unicodeToAscii('Ślusàrski')}")

converting 'Ślusàrski' to Slusarski


In [112]:
import os
import glob
from google.colab import drive


labels = []
names = []

drive.mount("/content/drive")
filenames = glob.glob('/content/drive/MyDrive/names/*.txt')
for filename in filenames:
  labels.append(os.path.splitext(os.path.basename(filename))[0])
  lines = open(filename, mode='r', encoding='utf8').read().strip().split('\n')
  names.extend([unicodeToAscii(line) for line in lines if unicodeToAscii(line).strip()])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [113]:
import torch
import string

eos = '\n'
special_chars = " .,;'-_"
all_letters = string.ascii_letters + special_chars + eos
vocab = {c:i for i, c in enumerate(all_letters)}

In [114]:
unique_chars = set("".join(char for char in list(name for name in names)))

In [118]:
unique_chars_str = ""
for i in range(len(unique_chars)):
  unique_chars_str += unique_chars.pop()

print(unique_chars_str)

In [119]:
def text_to_idx(text: str):
  return [vocab.get('_') if vocab.get(i) is None else vocab.get(i) for i in text]

def idx_to_text(idx) -> str:
  return "".join([all_letters[i] for i in idx])

In [120]:
def name_to_tensor(name: str) -> tuple:
  idx = text_to_idx(name + eos)
  x = torch.tensor(idx[:-1])
  y = torch.tensor(idx[1:])
  return x, y

In [121]:
import torch.nn as nn

class NameRNN(nn.Module):
  def __init__(self, vocab_size, hidden_size):
    super().__init__()
    self.emb = nn.Embedding(vocab_size, hidden_size)
    self.rnn = nn.RNN(hidden_size, hidden_size)
    self.out = nn.Linear(hidden_size, vocab_size)

  def forward(self, x):
    x = self.emb(x)
    output, hidden = self.rnn(x)
    logits = self.out(output)
    return logits

In [122]:
model = NameRNN(len(vocab), 128)

In [123]:
import random
import numpy as np

n_epochs = 10
batch_size = 4
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters())

model.train()
for epoch in range(n_epochs):

  indicies = list(range(len(names)))
  random.shuffle(indicies)
  batches = np.array_split(indicies, len(indicies) // batch_size)

  epoch_total_loss = 0

  for idx, batch_indices in enumerate(batches, 1):
    optimizer.zero_grad()

    current_batch_loss_tensor = 0
    for i in batch_indices:
      x, y = name_to_tensor(names[i])
      logits = model.forward(x)
      loss = criterion(logits, y)
      current_batch_loss_tensor += loss

    current_batch_loss_tensor.backward()
    optimizer.step()

    epoch_total_loss += current_batch_loss_tensor.item()

  avg_loss = epoch_total_loss / (idx * batch_size)
  print(f"epoch: {epoch + 1} loss: {avg_loss:.4f}")

epoch: 1 loss: 2.1918
epoch: 2 loss: 2.0308
epoch: 3 loss: 1.9714
epoch: 4 loss: 1.9362
epoch: 5 loss: 1.9102
epoch: 6 loss: 1.8901
epoch: 7 loss: 1.8744
epoch: 8 loss: 1.8634
epoch: 9 loss: 1.8526
epoch: 10 loss: 1.8425


In [222]:
start_token = "L"
start_token = (random.choice(string.ascii_uppercase) if start_token is None else start_token)
print(start_token)

L


In [235]:
def generate(num_samples=10, start_token=None):
  generated_names = []
  for i in range(num_samples):
    current_token = start_token if start_token else random.choice(string.ascii_uppercase)
    generated_name = current_token
    while True:
      x, _ = name_to_tensor(generated_name)
      logits = model.forward(x)
      pred = torch.softmax(logits[-1], dim=-1)
      token = torch.multinomial(pred, num_samples=1)
      generated_name += all_letters[token.item()]
      if token.item() == vocab[eos]:
        break
    generated_names.append(f"{i+1}. {generated_name}")
  return generated_names

print(*generate(15), sep='')

1. Arians
2. Pashour
3. Himilman
4. Hurasi
5. Power
6. Osellank
7. Ohaev
8. Quramaley
9. Cho
10. Otsinov
11. Uromchuz
12. Colwatco
13. Jakushin
14. Xura
15. Huirn

